In [1]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import bisect

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")

from pycbc.psd.analytical import aLIGO140MpcT1800545, aLIGO175MpcT1800545
from pycbc.filter import sigma, sigmasq

PyCBC.libutils: pkg-config call failed, setting NO_PKGCONFIG=1


In [2]:
import sys
import os

sys.path.insert(1, '../../detectability')

import resonance
from resonance import Filter, phase_diff_t_shift, phase_shift #System, 

/home/alberto/miniconda3/envs/gwsense/lib/python3.11/site-packages/pycbc/waveform/plugin.py:99: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
data_phi_path = '../data/III/snr/phi'
os.makedirs(data_phi_path, exist_ok=True) 
figures_phi_path = '../figures/III/snr/phi'
os.makedirs(figures_phi_path, exist_ok=True) 

data_t_path = '../data/III/snr/t'
os.makedirs(data_t_path, exist_ok=True) 
figures_t_path = '../figures/III/snr/t'
os.makedirs(figures_t_path, exist_ok=True) 

figures_compare_path = '../figures/III/snr/compare'
os.makedirs(figures_compare_path, exist_ok=True) 

# Detectability

In [4]:
def detectability_data(detector, srate, fres_arr, dPsi, dx_seed):
    filt.srate = srate
    filt.detector = detector
    print(detector)
    snr = 100
    MM_detect = 1/(2*snr)

    error = 1e-3
    dx_cut = filt.dx_fres_fix_error(error, fres_arr, dPsi)

    if dPsi==phase_shift:
        dx_det, dx_guess = filt.detectable_dx(MM_detect, fres_arr, dPsi, dx_seed, dx_cut)
    else:
        dx_det, dx_guess = filt.detectable_dx(MM_detect, fres_arr, dPsi, dx_seed, dx_cut)
    joc = np.array(filt.read2023_fixed_snr(snr))
    
    np.savez_compressed(os.path.join(f'../data/III/snr/{x_type[dPsi]}', f'{x_type[dPsi]}_{detector_name[detector]}.npz'), 
                        fres=fres_arr, detect=dx_det, guess=dx_guess, joc=joc, cut=dx_cut)


def snr_data(detector, snr, srate, fres_arr, dPsi, dx_seed):
    filt.srate = srate # ordering SNR update
    filt.detector = detector
    print(detector)
    MM_detect = 1/(2*snr**2)

    filt.tmplt_snr = filt.template_snr(m1,m2,snr)#filt.tmplt / sigma(filt.tmplt, **filt.kwargs) * snr
    

    #error = 1e-3
    #dx_cut = filt.dx_fres_fix_error(error, fres_arr, dPsi)

    if dPsi==phase_shift:
        dx_det, dx_guess = filt.detectable_dx(MM_detect, fres_arr, dPsi, dx_seed, None)
    else:
        dx_det, dx_guess = filt.detectable_dx(MM_detect, fres_arr, dPsi, dx_seed, None)
    joc = np.array(filt.read2023_fixed_snr(snr))
    
    # np.savez_compressed(os.path.join(f'../data/III/snr/{x_type[dPsi]}', f'{x_type[dPsi]}_{detector_name[detector]}.npz'), fres=fres_arr, detect=dx_det, guess=dx_guess, joc=joc, cut=dx_cut)

In [5]:
m1=1.4
m2=1.4

In [6]:
f_low = 15
f_high = 1024

duration = 512

srate = 16384

dL = 100
detector = 'aLIGO/Asharp_strain'



filt = Filter(f_low, f_high, duration, srate, dL, 'IMRPhenomD', detector) #f_low, f_high, tlen, srate, dL, approximant, detector

Requested number of samples exceeds the highest available frequency in the input data, will use max available frequency instead. (requested 8192.000000 Hz, available 6000.000000 Hz)
/home/alberto/miniconda3/envs/gwsense/lib/python3.11/site-packages/pycbc/types/array.py:390: RuntimeWarning: divide by zero encountered in divide
  return self._data.__rtruediv__(other)


In [7]:
detector_list = [aLIGO140MpcT1800545, aLIGO175MpcT1800545, 'aLIGO/AplusDesign', 'aLIGO/Asharp_strain', 'ET/ET_B', 'CE/CE_40']

detector_name = {detector_list[0]: 'O4', detector_list[1]: 'Aplus', detector_list[2]: 'Asharp', detector_list[3]: 'ET', detector_list[4]: 'CE'} 

#srate_list = [16384,16384,16384,16384,16384]

x_type = {phase_shift: 'phi', phase_diff_t_shift: 't'}

In [8]:
dpsi_seed = -1e-2
dt_seed = -1e-4


fres_arr = np.logspace(np.log10(20.), np.log10(500.), num=3)
snr_list = [12,36,100]

detector_list0 = [detector_list[0],detector_list[-1]]


In [9]:
detector = detector_list[1]
for snr in snr_list:    
    #template_snr(self, m1=1.4, m2=1.4, snr=100) 
    snr_data(detector, snr, srate, fres_arr, phase_diff_t_shift, dt_seed)

    snr_new_tmplt = sigma(filt.tmplt, **filt.kwargs)
    
    filt.dL = snr / snr_new_tmplt * filt.dL
    print(f'For a BNS at a luminosity distance of{filt.dL}, detected by detector {detector}, the SNR is {snr_new_tmplt}') 

<function aLIGO175MpcT1800545 at 0x765434f8b100>


/home/alberto/miniconda3/envs/gwsense/lib/python3.11/site-packages/pycbc/types/array.py:217: RuntimeWarning: invalid value encountered in divide
  ret = getattr(ufunc, method)(*inputs, **kwargs)
/home/alberto/miniconda3/envs/gwsense/lib/python3.11/site-packages/pycbc/types/array.py:383: RuntimeWarning: invalid value encountered in divide
  return self._data / other
/tmp/ipykernel_24221/1450906401.py:37: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  joc = np.array(filt.read2023_fixed_snr(snr))


For a BNS at a luminosity distance of37.88746276132567, detected by detector <function aLIGO175MpcT1800545 at 0x765434f8b100>, the SNR is 31.672746405835394
<function aLIGO175MpcT1800545 at 0x765434f8b100>
For a BNS at a luminosity distance of16.31577930739512, detected by detector <function aLIGO175MpcT1800545 at 0x765434f8b100>, the SNR is 83.59690540736324
<function aLIGO175MpcT1800545 at 0x765434f8b100>
For a BNS at a luminosity distance of8.404849108966978, detected by detector <function aLIGO175MpcT1800545 at 0x765434f8b100>, the SNR is 194.12340538021215


In [10]:
/

SyntaxError: invalid syntax (1303184558.py, line 1)

In [ ]:
#[detectability_data(detector, snr srate, fres_arr, phase_diff_t_shift, dt_seed) for snr in snr_list]

In [ ]:
#[detectability_data(detector, srate, fres_arr, phase_diff_t_shift, dt_seed) for srate, detector in zip(srate_list[:-1], detector_list[:-1])]

In [ ]:
#[detectability_data(detector, srate, fres_arr, phase_shift, dpsi_seed) for srate, detector in zip(srate_list, detector_list)]

detectability_data('CE/CE_40', 16384, fres_arr, phase_shift, -1e-2)
detectability_data('CE/CE_40', 16384, np.logspace(np.log10(20.), np.log10(450.), num=21), phase_diff_t_shift, -1e-4)